In [62]:
import pandas as pd
import numpy as np
import re
import os
import plotly.express as px
import plotly.graph_objects as go

# Paths
DATA_DIR = "data/"
OUTPUT_DIR = "outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

YEARS = [2023, 2024, 2025]

# Targets
TARGETS = {
    "HCVA": 150_000,
    "KTI": 0.8,
    "Skill_Decay": 0.1,
    "RE_Score": 4.2,
    "SPE": 0.3
}

# Weights
WEIGHTS = {
    "HCVA": 0.30,
    "KTI": 0.20,
    "Skill_Decay": 0.20,
    "RE_Score": 0.15,
    "SPE": 0.15
}

In [63]:
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]+", "_", regex=True)
    )
    return df

def normalize_text(s):
    if pd.isna(s):
        return s
    return str(s).strip().lower()

def filter_years(df):
    if "year" in df.columns:
        df = df[df["year"].isin(YEARS)]
    return df

def safe_div(a, b):
    return np.where((b == 0) | (pd.isna(b)), np.nan, a / b)

In [70]:
from pathlib import Path

base = Path("/Users/CoursM2/Documents/Alberthon/Sujet Alberthon")

finance = pd.read_excel(base / "Finance" / "AlbertSchool_CACEIS_PL-FTE_22-25_Sent.xlsx")

training = pd.read_excel(base / "Training" / "Training_Records_Unnamed.xlsx")

cold_review = pd.read_excel(base / "Training" / "Cold_Review_Unnamed.xlsx")

absenteeism = pd.read_excel(base / "HR Data" / "20260121 - Absentéisme_-_détail_affectation_-_Bilan_social 2025.xlsx")

In [71]:
# Nettoyage des colonnes
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]+", "_", regex=True)
    )
    return df

finance = clean_columns(finance)
training = clean_columns(training)
cold_review = clean_columns(cold_review)
absenteeism = clean_columns(absenteeism)

In [72]:
print("Finance columns:", finance.columns.tolist())
print("Training columns:", training.columns.tolist())
print("Absenteeism columns:", absenteeism.columns.tolist())

Finance columns: ['toutes_filiales_conso', 'réel_décembre_2022', 'réel_décembre_2023', 'réel_décembre_2024', 'réel_décembre_2025', 'unnamed_5', 'o_w_europe', 'réel_décembre_2022_1', 'réel_décembre_2023_1', 'réel_décembre_2024_1', 'réel_décembre_2025_1']
Training columns: ['employee_code', 'entity', 'direction', 'attended_courses', 'organization', 'seesion_start_date', 'session_end_date', 'session_id', 'status', 'total_training_hours', 'certifications', 'year']
Absenteeism columns: ['mat_compliance', 'employee_code', 'nom', 'prénom', 'genre', 'type_contrat', 'contrat_particulier', '_concat_contrat', 'présents_non_présents', 'code_régime_temps_travail', 'régime_temps_travail', 'régime_temps_travail_', 'code_société', 'société', 'code_niveau_6', 'niveau_6', 'code_niveau_7', 'niveau_7', 'code_niveau_8', 'niveau_8', 'code_niveau_9', 'niveau_9', 'code_organisation_1', 'organisation_1', 'code_organisation_2', 'organisation_2', 'code_organisation_3', 'organisation_3', 'code_organisation_4', 'o

In [73]:
# On sépare les deux blocs visibles :
# - Toutes Filiales Conso
# - O/W Europe

finance_raw = finance.copy()

finance_raw.head(10)

,toutes_filiales_conso,réel_décembre_2022,réel_décembre_2023,réel_décembre_2024,réel_décembre_2025,unnamed_5,o_w_europe,réel_décembre_2022_1,réel_décembre_2023_1,réel_décembre_2024_1,réel_décembre_2025_1
0,Net Commission Income,8.986520e+05,1.039248e+06,1.296794e+06,1.310191e+06,NaN,Net Commission Income,8.986520e+05,1.039441e+06,1.298118e+06,1.310091e+06
1,Net Interest Margin,4.186306e+05,6.944518e+05,8.265549e+05,8.676323e+05,NaN,Net Interest Margin,4.186306e+05,6.942633e+05,8.265514e+05,8.681490e+05
2,Other Income / (expense),-6.731757e+04,-5.636783e+04,-3.991230e+04,-7.781260e+04,NaN,Other Income / (expense),-6.731757e+04,-5.900319e+04,-4.145521e+04,-7.782268e+04
3,Net Banking Income (PNB),1.249965e+06,1.677332e+06,2.083437e+06,2.100011e+06,NaN,Net Banking Income (PNB),1.249965e+06,1.674701e+06,2.083214e+06,2.100417e+06
4,Rémunérations & charges,-3.923482e+05,-5.149449e+05,-6.580353e+05,-6.787855e+05,NaN,Rémunérations & charges,-3.923482e+05,-5.026284e+05,-6.309239e+05,-6.479100e+05
5,Recrutement,-3.141853e+03,-3.117929e+03,-3.199803e+03,-2.618171e+03,NaN,Recrutement,-3.141853e+03,-2.931983e+03,-2.860442e+03,-2.618171e+03
6,Formation (training costs),-4.385546e+03,-6.282893e+03,-5.924728e+03,-5.118793e+03,NaN,Formation (training costs),-4.385546e+03,-6.101923e+03,-5.654003e+03,-5.118793e+03
7,Other personnel costs,-6.056440e+04,-7.761735e+04,-9.776850e+04,-1.036153e+05,NaN,Other personnel costs,-6.056440e+04,-7.526670e+04,-9.172738e+04,-9.652365e+04
8,Total Personnel Costs,-4.604400e+05,-6.019631e+05,-7.649284e+05,-7.901378e+05,NaN,Total Personnel Costs,-4.604400e+05,-5.869290e+05,-7.311658e+05,-7.521706e+05
9,Other Operating Costs,-4.350758e+05,-5.814016e+05,-7.152383e+05,-6.502823e+05,NaN,Other Operating Costs,-4.350758e+05,-5.948595e+05,-7.520772e+05,-6.910448e+05


In [74]:
finance.columns = [
    "metric_conso",
    "2022_conso",
    "2023_conso",
    "2024_conso",
    "2025_conso",
    "separator",
    "metric_europe",
    "2022_europe",
    "2023_europe",
    "2024_europe",
    "2025_europe"
]

In [75]:
conso = finance[
    ["metric_conso", "2023_conso", "2024_conso", "2025_conso"]
].copy()

conso = conso.rename(columns={"metric_conso": "metric"})

conso["legal_entity"] = "Toutes Filiales Conso"

In [76]:
europe = finance[
    ["metric_europe", "2023_europe", "2024_europe", "2025_europe"]
].copy()

europe = europe.rename(columns={"metric_europe": "metric"})

europe["legal_entity"] = "O/W Europe"

In [77]:
finance_tidy = pd.concat([conso, europe], ignore_index=True)

finance_tidy = finance_tidy.melt(
    id_vars=["legal_entity", "metric"],
    value_vars=["2023_conso", "2024_conso", "2025_conso",
                "2023_europe", "2024_europe", "2025_europe"],
    var_name="year_source",
    value_name="value"
)

In [78]:
conso_long = conso.melt(
    id_vars=["legal_entity", "metric"],
    value_vars=["2023_conso", "2024_conso", "2025_conso"],
    var_name="year",
    value_name="value"
)

europe_long = europe.melt(
    id_vars=["legal_entity", "metric"],
    value_vars=["2023_europe", "2024_europe", "2025_europe"],
    var_name="year",
    value_name="value"
)

finance_tidy = pd.concat([conso_long, europe_long], ignore_index=True)

finance_tidy["year"] = finance_tidy["year"].str.extract(r"(\d{4})").astype(int)

In [79]:
finance_tidy["value"] = (
    finance_tidy["value"]
    .astype(str)
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
)

finance_tidy["value"] = pd.to_numeric(finance_tidy["value"], errors="coerce")

In [80]:
finance_tidy["metric"].dropna().unique()

array(['Net Commission Income', 'Net Interest Margin',
       'Other Income / (expense)', 'Net Banking Income (PNB)',
       'Rémunérations & charges', 'Recrutement',
       'Formation (training costs)', 'Other  personnel costs',
       'Total Personnel Costs', 'Other Operating Costs',
       'Total Operating Costs', 'Gross Operating Income (RBE)'],
      dtype=object)

In [81]:
hcva_input = finance_tidy[
    finance_tidy["metric"].isin([
        "Net Banking Income (PNB)",
        "Total Operating Costs",
        "Rémunérations & charges"
    ])
].copy()

hcva_input = hcva_input.pivot_table(
    index=["year", "legal_entity"],
    columns="metric",
    values="value",
    aggfunc="sum"
).reset_index()

hcva_input.columns.name = None

hcva_input = hcva_input.rename(columns={
    "Net Banking Income (PNB)": "gnp",
    "Total Operating Costs": "operating_costs",
    "Rémunérations & charges": "remuneration"
})

hcva_input

,year,legal_entity,gnp,remuneration,operating_costs
0,2023,O/W Europe,1.674701e+06,-502628.44068,-1.181789e+06
1,2023,Toutes Filiales Conso,1.677332e+06,-514944.93302,-1.183365e+06
2,2024,O/W Europe,2.083214e+06,-630923.92936,-1.483243e+06
3,2024,Toutes Filiales Conso,2.083437e+06,-658035.31625,-1.480167e+06
4,2025,O/W Europe,2.100417e+06,-647910.00779,-1.443215e+06
5,2025,Toutes Filiales Conso,2.100011e+06,-678785.49119,-1.440420e+06


In [82]:
hcva_input["remuneration"] = hcva_input["remuneration"].abs()
hcva_input["operating_costs"] = hcva_input["operating_costs"].abs()

In [84]:
# Temporary placeholder until FTE sheet/table is integrated
hcva_input["fte"] = np.nan
hcva_input["direction"] = "All Directions"

In [85]:
hcva_input["remuneration"] = hcva_input["remuneration"].abs()
hcva_input["operating_costs"] = hcva_input["operating_costs"].abs()

hcva_input["hcva"] = (
    hcva_input["gnp"] - 
    (hcva_input["operating_costs"] - hcva_input["remuneration"])
) / hcva_input["fte"]

hcva_df = hcva_input[
    ["year", "legal_entity", "direction", "hcva"]
]

hcva_df

,year,legal_entity,direction,hcva
0,2023,O/W Europe,All Directions,NaN
1,2023,Toutes Filiales Conso,All Directions,NaN
2,2024,O/W Europe,All Directions,NaN
3,2024,Toutes Filiales Conso,All Directions,NaN
4,2025,O/W Europe,All Directions,NaN
5,2025,Toutes Filiales Conso,All Directions,NaN


In [86]:
xls = pd.ExcelFile(base / "Finance" / "AlbertSchool_CACEIS_PL-FTE_22-25_Sent.xlsx")
print(xls.sheet_names)

['Synthese_PL', 'Synthese_ETP']


In [87]:
fte_raw = pd.read_excel(
    base / "Finance" / "AlbertSchool_CACEIS_PL-FTE_22-25_Sent.xlsx",
    sheet_name="Synthese_ETP"
)

fte_raw = clean_columns(fte_raw)

fte_raw.head()

,employee_numbers_end_of_year_average_for_year,unnamed_1,réel_décembre_2022,unnamed_3,réel_décembre_2023,unnamed_5,réel_décembre_2024,unnamed_7,réel_décembre_2025,unnamed_9
0,NaN,NaN,ETP fin de période,ETP moyen période,ETP fin de période,ETP moyen période,ETP fin de période,ETP moyen période,ETP fin de période,ETP moyen période
1,TOTAL (Toutes Filiales conso),NaN,3990.94,3963.56,6395.34,6370.66,6616.4,6635.56,6453.61,6631.47
2,NaN,CDI - MAD - FON,3699.66,3634.01,6018.39,5980.2,6033.37,6122.87,5953.64,5992.12
3,NaN,CDD,291.28,329.55,376.95,390.46,583.03,512.69,499.97,639.35
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [88]:
fte_tidy = fte_raw.copy()

# Renommer les colonnes utiles
fte_tidy = fte_tidy.rename(columns={
    "employee_numbers_end_of_year_average_for_year": "legal_entity",
    "unnamed_5": "fte_2023",
    "unnamed_7": "fte_2024",
    "unnamed_9": "fte_2025"
})

# Garder uniquement les lignes avec une entité légale
fte_tidy = fte_tidy[
    fte_tidy["legal_entity"].notna()
].copy()

# Garder uniquement les colonnes nécessaires
fte_tidy = fte_tidy[
    ["legal_entity", "fte_2023", "fte_2024", "fte_2025"]
]

# Supprimer la ligne header si elle existe
fte_tidy = fte_tidy[
    ~fte_tidy["legal_entity"].astype(str).str.contains("employee|nan", case=False, na=False)
]

fte_tidy

,legal_entity,fte_2023,fte_2024,fte_2025
1,TOTAL (Toutes Filiales conso),6370.66,6635.56,6631.47
5,o/w TOTAL (Europe),5338.66,5459.56,5425.46


In [89]:
fte_df = fte_tidy.melt(
    id_vars="legal_entity",
    value_vars=["fte_2023", "fte_2024", "fte_2025"],
    var_name="year",
    value_name="fte"
)

fte_df["year"] = fte_df["year"].str.extract(r"(\d{4})").astype(int)
fte_df["fte"] = pd.to_numeric(fte_df["fte"], errors="coerce")

fte_df

,legal_entity,year,fte
0,TOTAL (Toutes Filiales conso),2023,6370.66
1,o/w TOTAL (Europe),2023,5338.66
2,TOTAL (Toutes Filiales conso),2024,6635.56
3,o/w TOTAL (Europe),2024,5459.56
4,TOTAL (Toutes Filiales conso),2025,6631.47
5,o/w TOTAL (Europe),2025,5425.46


In [90]:
fte_df["legal_entity"] = (
    fte_df["legal_entity"]
    .astype(str)
    .str.replace("TOTAL \\(", "", regex=True)
    .str.replace("\\)", "", regex=True)
    .str.strip()
)

fte_df

,legal_entity,year,fte
0,Toutes Filiales conso,2023,6370.66
1,o/w Europe,2023,5338.66
2,Toutes Filiales conso,2024,6635.56
3,o/w Europe,2024,5459.56
4,Toutes Filiales conso,2025,6631.47
5,o/w Europe,2025,5425.46


In [91]:
hcva_input = hcva_input.drop(columns=["fte"], errors="ignore")

hcva_input = hcva_input.merge(
    fte_df,
    on=["year", "legal_entity"],
    how="left"
)

hcva_input

,year,legal_entity,gnp,remuneration,operating_costs,direction,hcva,fte
0,2023,O/W Europe,1.674701e+06,502628.44068,1.181789e+06,All Directions,NaN,NaN
1,2023,Toutes Filiales Conso,1.677332e+06,514944.93302,1.183365e+06,All Directions,NaN,NaN
2,2024,O/W Europe,2.083214e+06,630923.92936,1.483243e+06,All Directions,NaN,NaN
3,2024,Toutes Filiales Conso,2.083437e+06,658035.31625,1.480167e+06,All Directions,NaN,NaN
4,2025,O/W Europe,2.100417e+06,647910.00779,1.443215e+06,All Directions,NaN,NaN
5,2025,Toutes Filiales Conso,2.100011e+06,678785.49119,1.440420e+06,All Directions,NaN,NaN


In [92]:
hcva_input["remuneration"] = hcva_input["remuneration"].abs()
hcva_input["operating_costs"] = hcva_input["operating_costs"].abs()

hcva_input["hcva"] = (
    hcva_input["gnp"] - 
    (hcva_input["operating_costs"] - hcva_input["remuneration"])
) / hcva_input["fte"]

hcva_input["direction"] = "All Directions"

hcva_df = hcva_input[
    ["year", "legal_entity", "direction", "hcva"]
]

hcva_df

,year,legal_entity,direction,hcva
0,2023,O/W Europe,All Directions,NaN
1,2023,Toutes Filiales Conso,All Directions,NaN
2,2024,O/W Europe,All Directions,NaN
3,2024,Toutes Filiales Conso,All Directions,NaN
4,2025,O/W Europe,All Directions,NaN
5,2025,Toutes Filiales Conso,All Directions,NaN


In [93]:
print("HCVA entities:")
print(hcva_input["legal_entity"].unique())

print("\nFTE entities:")
print(fte_df["legal_entity"].unique())

HCVA entities:
['O/W Europe' 'Toutes Filiales Conso']

FTE entities:
['Toutes Filiales conso' 'o/w Europe']


In [94]:
def normalize_entity(x):
    x = str(x).strip().lower()
    x = x.replace("total", "")
    x = x.replace("(", "").replace(")", "")
    x = x.replace("toutes filiales conso", "toutes filiales conso")
    x = x.replace("o/w europe", "o/w europe")
    return x.strip()

hcva_input["entity_key"] = hcva_input["legal_entity"].apply(normalize_entity)
fte_df["entity_key"] = fte_df["legal_entity"].apply(normalize_entity)

In [95]:
hcva_input = hcva_input.drop(columns=["fte"], errors="ignore")

hcva_input = hcva_input.merge(
    fte_df[["year", "entity_key", "fte"]],
    on=["year", "entity_key"],
    how="left"
)

hcva_input[["year", "legal_entity", "fte"]]

,year,legal_entity,fte
0,2023,O/W Europe,5338.66
1,2023,Toutes Filiales Conso,6370.66
2,2024,O/W Europe,5459.56
3,2024,Toutes Filiales Conso,6635.56
4,2025,O/W Europe,5425.46
5,2025,Toutes Filiales Conso,6631.47


In [96]:
hcva_input["remuneration"] = hcva_input["remuneration"].abs()
hcva_input["operating_costs"] = hcva_input["operating_costs"].abs()

hcva_input["hcva"] = (
    hcva_input["gnp"] - 
    (hcva_input["operating_costs"] - hcva_input["remuneration"])
) / hcva_input["fte"]

hcva_df = hcva_input[
    ["year", "legal_entity", "direction", "hcva"]
]

hcva_df

,year,legal_entity,direction,hcva
0,2023,O/W Europe,All Directions,186.477702
1,2023,Toutes Filiales Conso,All Directions,158.368498
2,2024,O/W Europe,All Directions,225.456813
3,2024,Toutes Filiales Conso,All Directions,190.082728
4,2025,O/W Europe,All Directions,240.553225
5,2025,Toutes Filiales Conso,All Directions,201.821923


In [97]:
cold_review.head()


,date,matricule,formation,organization,session_id,date_de_début_de_session,date_de_fin_de_session,lieu_de_session,status,considérez_vous_que_cette_formation_vous_a_permis_de_prendre_confiance_en_vous_,considérez_vous_que_cette_formation_vous_a_permis_de_faciliter_votre_quotidien_,considérez_vous_que_cette_formation_vous_a_permis_d_améliorer_la_qualité_ou_l_efficacité_de_votre_travail_,considérez_vous_que_cette_formation_vous_a_permis_de_vous_perfectionner_dans_un_domaine_que_vous_connaissiez_déjà_,considérez_vous_que_cette_formation_vous_a_permis_de_développer_de_nouvelles_compétences_,autres_à_préciser_,la_formation_visait_elle_la_préparation_d_un_diplôme_ou_d_une_certification_,si_oui_avez_vous_obtenu_le_diplôme_ou_la_certification_visé_e_,si_non_pourquoi_,la_formation_a_t_elle_répondu_à_vos_attentes_initiales_,estimez_vous_que_la_formation_était_en_adéquation_avec_le_métier_ou_les_réalités_du_secteur_,recommanderiez_vous_ce_stage_à_une_personne_exerçant_le_même_métier_que_vous_,utilisez_vous_les_connaissances_acquises_lors_de_la_formation_,quels_étaient_selon_vous_les_principaux_points_forts_et_les_principaux_axes_d_amélioration_de_cette_formation_
0,19/02/2026,ANON_76X90X48X48X57X49X50X55X55X48X53X,Formation FinOps : Maîtriser et optimiser ses dépenses sur AWS,PLB CONSULTANT,25-FF-724-572,20/11/2025,21/11/2025,A distance,En attente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19/02/2026,ANON_76X90X48X48X57X49X53X52X48X56X52X,"Bâle 3, quels enjeux pour les métiers de la finance ?",IFCAM,25-B3Q-739-075,20/11/2025,21/11/2025,Paris,Complétée,Oui,Oui,Oui,Oui,Oui,NaN,Non,NaN,NaN,"Oui, tout à fait","Oui, tout à fait","Oui, tout à fait","Oui, en partie",NaN
2,18/02/2026,ANON_76X90X48X48X57X49X50X52X49X52X56X,"Parcours nouveaux managers - Suivre, piloter et organiser le travail de son équipe",ERYS,25-PNM-889-284,20/11/2025,20/11/2025,Montrouge,En attente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,18/02/2026,ANON_76X90X48X48X57X49X49X55X49X56X57X,"Parcours nouveaux managers - Suivre, piloter et organiser le travail de son équipe",ERYS,25-PNM-889-284,20/11/2025,20/11/2025,Montrouge,Complétée,Oui,Oui,Oui,Oui,Oui,ok,Non,Non,ok,"Oui, en partie","Oui, en partie","Oui, en partie","Oui, en partie",NaN
4,18/02/2026,ANON_75X88X48X48X48X48X50X55X56X55X55X,"Parcours nouveaux managers - Suivre, piloter et organiser le travail de son équipe",ERYS,25-PNM-889-284,20/11/2025,20/11/2025,Montrouge,En attente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [98]:
cold_review.columns.tolist()

['date',
 'matricule',
 'formation',
 'organization',
 'session_id',
 'date_de_début_de_session',
 'date_de_fin_de_session',
 'lieu_de_session',
 'status',
 'considérez_vous_que_cette_formation_vous_a_permis_de_prendre_confiance_en_vous_',
 'considérez_vous_que_cette_formation_vous_a_permis_de_faciliter_votre_quotidien_',
 'considérez_vous_que_cette_formation_vous_a_permis_d_améliorer_la_qualité_ou_l_efficacité_de_votre_travail_',
 'considérez_vous_que_cette_formation_vous_a_permis_de_vous_perfectionner_dans_un_domaine_que_vous_connaissiez_déjà_',
 'considérez_vous_que_cette_formation_vous_a_permis_de_développer_de_nouvelles_compétences_',
 'autres_à_préciser_',
 'la_formation_visait_elle_la_préparation_d_un_diplôme_ou_d_une_certification_',
 'si_oui_avez_vous_obtenu_le_diplôme_ou_la_certification_visé_e_',
 'si_non_pourquoi_',
 'la_formation_a_t_elle_répondu_à_vos_attentes_initiales_',
 'estimez_vous_que_la_formation_était_en_adéquation_avec_le_métier_ou_les_réalités_du_secteur_',
 'r

In [100]:
training.head()

,employee_code,entity,direction,attended_courses,organization,seesion_start_date,session_end_date,session_id,status,total_training_hours,certifications,year
0,NaN,NaN,NaN,Modèle de données GP4,NEOXAM,15/10/2025,17/10/2025,25-MDG-859-226,Réalisé,21.0,No,2025
1,NaN,CACEIS Bank,NaN,2024 International compliance trainings,IFCAM,29/11/2024,29/11/2024,NaN,Annulée,0.0,No,2024
2,NaN,CACEIS Bank,NaN,7Speaking plateforme licences annuelles,7Speaking,2025-06-01 00:00:00,2025-06-01 00:00:00,24-7PL-451-669,Réalisé,0.5,No,2025
3,NaN,CACEIS Bank,NaN,7Speaking plateforme licences annuelles,7Speaking,15/12/2023,15/12/2023,23-7PL-451-343,Réalisé,0.1,NaN,2023
4,NaN,CACEIS Bank,NaN,Accompagnement à la négociation commerciale,Dale Carnegie,NaN,NaN,NaN,Annulée,14.0,No,2025


In [101]:
training_clean = training.copy()

training_clean["status"] = training_clean["status"].str.lower()
training_clean["entity"] = training_clean["entity"].fillna("Unknown")
training_clean["direction"] = training_clean["direction"].fillna("Unknown")

In [102]:
training_clean["is_completed"] = training_clean["status"].isin(["réalisé", "realise"])

utilization = training_clean.groupby(
    ["year", "entity", "direction"]
).agg(
    completed=("is_completed", "sum"),
    total=("is_completed", "count")
).reset_index()

utilization["utilization_score"] = utilization["completed"] / utilization["total"]

utilization

,year,entity,direction,completed,total,utilization_score
0,2023,CACEIS,BUT - Business Units & Tech,2,2,1.000000
1,2023,CACEIS,BUT - Inf System Sec & Resil,1,1,1.000000
2,2023,CACEIS,BUT - Market Solutions,1,1,1.000000
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,1,1,1.000000
4,2023,CACEIS,COV - Client & Bus Dev Support,4,4,1.000000
5,2023,CACEIS,COV - PERES,7,7,1.000000
6,2023,CACEIS,FINANCE AND ADMINISTRATION,5,6,0.833333
7,2023,CACEIS,GENERAL INSPECTION,11,13,0.846154
8,2023,CACEIS,General Management,14,16,0.875000
9,2023,CACEIS,Human Resources,30,32,0.937500


In [103]:
active_users = training_clean.groupby(
    ["year", "entity", "direction"]
)["employee_code"].nunique().reset_index(name="active_users")

total_users = training_clean.groupby(
    ["year", "entity", "direction"]
)["employee_code"].count().reset_index(name="total_records")

response = active_users.merge(total_users,
                              on=["year", "entity", "direction"])

response["response_rate"] = response["active_users"] / response["total_records"]

response

,year,entity,direction,active_users,total_records,response_rate
0,2023,CACEIS,BUT - Business Units & Tech,1,2,0.500000
1,2023,CACEIS,BUT - Inf System Sec & Resil,1,1,1.000000
2,2023,CACEIS,BUT - Market Solutions,1,1,1.000000
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,1,1,1.000000
4,2023,CACEIS,COV - Client & Bus Dev Support,1,4,0.250000
5,2023,CACEIS,COV - PERES,1,7,0.142857
6,2023,CACEIS,FINANCE AND ADMINISTRATION,4,6,0.666667
7,2023,CACEIS,GENERAL INSPECTION,3,13,0.230769
8,2023,CACEIS,General Management,6,16,0.375000
9,2023,CACEIS,Human Resources,7,32,0.218750


In [104]:
training_clean = training.copy()

training_clean["status"] = training_clean["status"].astype(str).str.lower().str.strip()
training_clean["entity"] = training_clean["entity"].fillna("Unknown")
training_clean["direction"] = training_clean["direction"].fillna("Unknown")

training_clean = training_clean[
    (training_clean["year"].isin([2023, 2024, 2025])) &
    (training_clean["entity"] != "Unknown") &
    (training_clean["direction"] != "Unknown")
].copy()

training_clean["is_completed"] = training_clean["status"].isin([
    "réalisé",
    "realise",
    "réalisée",
    "completed",
    "done"
])

In [105]:
utilization = training_clean.groupby(
    ["year", "entity", "direction"]
).agg(
    completed=("is_completed", "sum"),
    total_sessions=("is_completed", "count")
).reset_index()

utilization["utilization_score"] = utilization["completed"] / utilization["total_sessions"]

In [106]:
response = training_clean.groupby(
    ["year", "entity", "direction"]
).agg(
    active_users=("employee_code", "nunique"),
    total_records=("employee_code", "count")
).reset_index()

response["response_rate"] = response["active_users"] / response["total_records"]

In [107]:
kti_df = utilization.merge(
    response,
    on=["year", "entity", "direction"],
    how="inner"
)

kti_df["kti"] = kti_df["utilization_score"] * kti_df["response_rate"]

kti_df = kti_df.rename(columns={"entity": "legal_entity"})

kti_df = kti_df[
    ["year", "legal_entity", "direction", "utilization_score", "response_rate", "kti"]
]

kti_df.head()

,year,legal_entity,direction,utilization_score,response_rate,kti
0,2023,CACEIS,BUT - Business Units & Tech,1.0,0.50,0.50
1,2023,CACEIS,BUT - Inf System Sec & Resil,1.0,1.00,1.00
2,2023,CACEIS,BUT - Market Solutions,1.0,1.00,1.00
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,1.0,1.00,1.00
4,2023,CACEIS,COV - Client & Bus Dev Support,1.0,0.25,0.25


In [108]:
kti_df["kti"].describe()

count    166.000000
mean       0.399345
std        0.272565
min        0.102041
25%        0.206772
50%        0.326904
75%        0.444444
max        1.000000
Name: kti, dtype: float64

In [109]:
kti_df = kti_df.copy()

# Reset index propre
kti_df = kti_df.reset_index(drop=True)

# Supprimer lignes corrompues
kti_df = kti_df[
    (kti_df["year"].isin([2023, 2024, 2025])) &
    (kti_df["year"].notna())
]

# Supprimer doublons exacts
kti_df = kti_df.drop_duplicates()

In [110]:
kti_df = kti_df.groupby(
    ["year", "legal_entity", "direction"]
).agg(
    kti=("kti", "mean"),
    utilization_score=("utilization_score", "mean"),
    response_rate=("response_rate", "mean")
).reset_index()

In [111]:
kti_df.head(10)

,year,legal_entity,direction,kti,utilization_score,response_rate
0,2023,CACEIS,BUT - Business Units & Tech,0.500000,1.000000,0.500000
1,2023,CACEIS,BUT - Inf System Sec & Resil,1.000000,1.000000,1.000000
2,2023,CACEIS,BUT - Market Solutions,1.000000,1.000000,1.000000
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,1.000000,1.000000,1.000000
4,2023,CACEIS,COV - Client & Bus Dev Support,0.250000,1.000000,0.250000
5,2023,CACEIS,COV - PERES,0.142857,1.000000,0.142857
6,2023,CACEIS,FINANCE AND ADMINISTRATION,0.555556,0.833333,0.666667
7,2023,CACEIS,GENERAL INSPECTION,0.195266,0.846154,0.230769
8,2023,CACEIS,General Management,0.328125,0.875000,0.375000
9,2023,CACEIS,Human Resources,0.205078,0.937500,0.218750


In [112]:
kti_df.duplicated(
    ["year", "legal_entity", "direction"]
).sum()

np.int64(0)

In [113]:
training_sd = training.copy()

training_sd["seesion_start_date"] = pd.to_datetime(
    training_sd["seesion_start_date"],
    errors="coerce"
)

training_sd["year"] = training_sd["year"].astype(int)

In [114]:
last_training = training_sd.groupby(
    ["employee_code", "entity", "direction"]
)["seesion_start_date"].max().reset_index()

last_training.head()

,employee_code,entity,direction,seesion_start_date
0,ANON_65X48X48X48X48X50X51X54X48X55X50X,CACEIS,SPF - Corporate Compliance,2025-04-16
1,ANON_65X48X48X48X48X50X51X55X53X55X53X,CACEIS,SPF - Risk & Permanent Controls,2024-01-15
2,ANON_65X48X48X48X48X50X51X57X56X56X53X,CACEIS Bank,BUT - Information Technology,2024-10-01
3,ANON_65X48X48X48X50X53X48X49X53X52X48X,CACEIS Bank,BUT - Information Technology,2025-03-13
4,ANON_65X48X48X48X50X53X48X51X49X50X57X,CACEIS Bank,STI - ESG,2025-08-10


In [115]:
last_training = last_training.merge(
    training_sd[["employee_code", "year"]],
    on="employee_code",
    how="left"
)

last_training["reference_date"] = pd.to_datetime(
    last_training["year"].astype(str) + "-12-31"
)

In [116]:
last_training["months_since_training"] = (
    (last_training["reference_date"] - last_training["seesion_start_date"])
    .dt.days / 30
)

In [117]:
last_training["decay_flag"] = last_training["months_since_training"] > 18

In [118]:
skill_decay_df = last_training.groupby(
    ["year", "entity", "direction"]
).agg(
    decay=("decay_flag", "sum"),
    total=("decay_flag", "count")
).reset_index()

skill_decay_df["skill_decay"] = skill_decay_df["decay"] / skill_decay_df["total"]

skill_decay_df = skill_decay_df.rename(columns={
    "entity": "legal_entity"
})

skill_decay_df = skill_decay_df[
    ["year", "legal_entity", "direction", "skill_decay"]
]

skill_decay_df.head()

,year,legal_entity,direction,skill_decay
0,2023,CACEIS,BUT - Business Units & Tech,0.0
1,2023,CACEIS,BUT - Inf System Sec & Resil,0.0
2,2023,CACEIS,BUT - Market Solutions,0.0
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,0.0
4,2023,CACEIS,COV - Client & Bus Dev Support,0.0


In [119]:
training_sd = training.copy()

training_sd["seesion_start_date"] = pd.to_datetime(
    training_sd["seesion_start_date"],
    errors="coerce",
    dayfirst=True
)

training_sd = training_sd[
    (training_sd["year"].isin([2023, 2024, 2025])) &
    (training_sd["employee_code"].notna()) &
    (training_sd["entity"].notna()) &
    (training_sd["direction"].notna()) &
    (training_sd["seesion_start_date"].notna())
].copy()

In [120]:
skill_rows = []

for year in [2023, 2024, 2025]:
    ref_date = pd.Timestamp(f"{year}-12-31")

    temp = training_sd[
        training_sd["seesion_start_date"] <= ref_date
    ].copy()

    last_training_year = temp.groupby(
        ["employee_code", "entity", "direction"]
    )["seesion_start_date"].max().reset_index()

    last_training_year["year"] = year

    last_training_year["months_since_training"] = (
        (ref_date - last_training_year["seesion_start_date"]).dt.days / 30
    )

    last_training_year["decay_flag"] = (
        last_training_year["months_since_training"] > 18
    )

    skill_rows.append(last_training_year)

skill_base = pd.concat(skill_rows, ignore_index=True)

In [121]:
skill_decay_df = skill_base.groupby(
    ["year", "entity", "direction"]
).agg(
    decay=("decay_flag", "sum"),
    total=("decay_flag", "count")
).reset_index()

skill_decay_df["skill_decay"] = (
    skill_decay_df["decay"] / skill_decay_df["total"]
)

skill_decay_df = skill_decay_df.rename(columns={
    "entity": "legal_entity"
})

skill_decay_df = skill_decay_df[
    ["year", "legal_entity", "direction", "skill_decay"]
]

skill_decay_df.head(10)

,year,legal_entity,direction,skill_decay
0,2023,CACEIS,BUT - Business Units & Tech,0.0
1,2023,CACEIS,BUT - Inf System Sec & Resil,0.0
2,2023,CACEIS,BUT - Market Solutions,0.0
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,0.0
4,2023,CACEIS,COV - Client & Bus Dev Support,0.0
5,2023,CACEIS,COV - PERES,0.0
6,2023,CACEIS,FINANCE AND ADMINISTRATION,0.0
7,2023,CACEIS,GENERAL INSPECTION,0.0
8,2023,CACEIS,General Management,0.0
9,2023,CACEIS,Human Resources,0.0


In [122]:
skill_decay_df["skill_decay"].describe()

count    209.000000
mean       0.215065
std        0.344079
min        0.000000
25%        0.000000
50%        0.000000
75%        0.333333
max        1.000000
Name: skill_decay, dtype: float64

In [123]:
skill_decay_df.sort_values("skill_decay", ascending=False).head(10)

,year,legal_entity,direction,skill_decay
150,2025,CACEIS,PERES GLOBAL SERVICES,1.0
183,2025,CACEIS Bank,ORGANISATION AND TRANSFORMATION,1.0
71,2024,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,1.0
169,2025,CACEIS Bank,COMPLIANCE,1.0
145,2025,CACEIS,FINANCE AND ADMINISTRATION,1.0
146,2025,CACEIS,GENERAL INSPECTION,1.0
185,2025,CACEIS Bank,PROCUREMENT AND NETWORK,1.0
184,2025,CACEIS Bank,PERES,1.0
182,2025,CACEIS Bank,OPERATIONS,1.0
135,2024,CACEIS Fund Administration,PERES,1.0


In [124]:
abs_clean = absenteeism.copy()

abs_clean["year"] = (
    abs_clean["aaaa_mm_absence"]
    .astype(str)
    .str[:4]
    .astype(int)
)

abs_clean = abs_clean[
    abs_clean["year"].isin([2023, 2024, 2025])
].copy()

abs_clean["legal_entity"] = abs_clean["société"]
abs_clean["direction"] = abs_clean["organisation_3"]

abs_clean["jours_ouvrés_absence"] = pd.to_numeric(
    abs_clean["jours_ouvrés_absence"],
    errors="coerce"
)

abs_clean.head()

,mat_compliance,employee_code,nom,prénom,genre,type_contrat,contrat_particulier,_concat_contrat,présents_non_présents,code_régime_temps_travail,régime_temps_travail,régime_temps_travail_,code_société,société,code_niveau_6,niveau_6,code_niveau_7,niveau_7,code_niveau_8,niveau_8,code_niveau_9,niveau_9,code_organisation_1,organisation_1,code_organisation_2,organisation_2,code_organisation_3,organisation_3,code_organisation_4,organisation_4,code_organisation_5,organisation_5,code_organisation_6,organisation_6,code_organisation_7,organisation_7,code_organisation_8,organisation_8,date_absence,aaaa_mm_absence,code_motif_jour_absence,motif_jour_absence,regroupement_jour_absences,jour_calendaires_absence,jours_ouvrables_absence,jours_ouvrés_absence,year,legal_entity,direction
0,NaN,ANON_73X74X48X49X49X57X53X51X54X51X50X,XX,XX,Masculin,CDI,NC,CDINCStandard,Présents,2,Cadre hors classe,Cadres,349,CACEIS Bank,BUI2A,BUT - Information Technology,I2COB,IT Coverage,I234C,IT Coverage,I2A0S,Client demands,CACEIS,CACEIS,B,CACEIS BANK,BBUI2A,BUT - Information Technology,BBUI2AI2COB,IT Coverage,BBUI2AI2COBI234C,IT Coverage,BBUI2AI2COBI234CI2A0S,Client demands,NC,NaN,NC,NaN,2025-05-02,2025/05,RECUP,Récupération,Congés,1.0,1.0,1.0,2025,CACEIS Bank,BUT - Information Technology
1,NaN,ANON_73X74X48X49X49X57X53X51X54X51X50X,XX,XX,Masculin,CDI,NC,CDINCStandard,Présents,2,Cadre hors classe,Cadres,349,CACEIS Bank,BUI2A,BUT - Information Technology,I2COB,IT Coverage,I234C,IT Coverage,I2A0S,Client demands,CACEIS,CACEIS,B,CACEIS BANK,BBUI2A,BUT - Information Technology,BBUI2AI2COB,IT Coverage,BBUI2AI2COBI234C,IT Coverage,BBUI2AI2COBI234CI2A0S,Client demands,NC,NaN,NC,NaN,2025-05-30,2025/05,RECUP,Récupération,Congés,1.0,1.0,1.0,2025,CACEIS Bank,BUT - Information Technology
2,NaN,ANON_73X74X48X49X49X57X53X51X54X51X50X,XX,XX,Masculin,CDI,NC,CDINCStandard,Présents,2,Cadre hors classe,Cadres,349,CACEIS Bank,BUI2A,BUT - Information Technology,I2COB,IT Coverage,I234C,IT Coverage,I2A0S,Client demands,CACEIS,CACEIS,B,CACEIS BANK,BBUI2A,BUT - Information Technology,BBUI2AI2COB,IT Coverage,BBUI2AI2COBI234C,IT Coverage,BBUI2AI2COBI234CI2A0S,Client demands,NC,NaN,NC,NaN,2025-06-20,2025/06,RECUP,Récupération,Congés,1.0,1.0,1.0,2025,CACEIS Bank,BUT - Information Technology
3,NaN,ANON_73X74X48X49X49X57X53X51X54X51X50X,XX,XX,Masculin,CDI,NC,CDINCStandard,Présents,2,Cadre hors classe,Cadres,349,CACEIS Bank,BUI2A,BUT - Information Technology,I2COB,IT Coverage,I234C,IT Coverage,I2A0S,Client demands,CACEIS,CACEIS,B,CACEIS BANK,BBUI2A,BUT - Information Technology,BBUI2AI2COB,IT Coverage,BBUI2AI2COBI234C,IT Coverage,BBUI2AI2COBI234CI2A0S,Client demands,NC,NaN,NC,NaN,2025-07-11,2025/07,CONGP,Congés annuels,Congés,1.0,1.0,1.0,2025,CACEIS Bank,BUT - Information Technology
4,NaN,ANON_73X74X48X49X49X57X53X51X54X51X50X,XX,XX,Masculin,CDI,NC,CDINCStandard,Présents,2,Cadre hors classe,Cadres,349,CACEIS Bank,BUI2A,BUT - Information Technology,I2COB,IT Coverage,I234C,IT Coverage,I2A0S,Client demands,CACEIS,CACEIS,B,CACEIS BANK,BBUI2A,BUT - Information Technology,BBUI2AI2COB,IT Coverage,BBUI2AI2COBI234C,IT Coverage,BBUI2AI2COBI234CI2A0S,Client demands,NC,NaN,NC,NaN,2025-07-12,2025/07,CONGP,Congés annuels,Congés,1.0,1.0,0.0,2025,CACEIS Bank,BUT - Information Technology


In [125]:
abs_df = abs_clean.groupby(
    ["year", "legal_entity", "direction"]
).agg(
    absent_days=("jours_ouvrés_absence", "sum"),
    employees_absent=("employee_code", "nunique")
).reset_index()

abs_df["absenteeism_rate"] = abs_df["absent_days"] / (abs_df["employees_absent"] * 220)

abs_df = abs_df.replace([np.inf, -np.inf], np.nan)

abs_df.head()

,year,legal_entity,direction,absent_days,employees_absent,absenteeism_rate
0,2025,CACEIS,BUT - Business Units & Tech,36.0,1,0.163636
1,2025,CACEIS,BUT - Gen Secretary & Controls,29.5,1,0.134091
2,2025,CACEIS,BUT - Inf System Sec & Resil,36.0,1,0.163636
3,2025,CACEIS,BUT - Information Technology,26.0,1,0.118182
4,2025,CACEIS,BUT - Market Solutions,33.0,1,0.150000


In [127]:
engagement_df = kti_df.copy()

engagement_df["engagement_score"] = engagement_df["kti"] * 5

engagement_df = engagement_df[
    ["year", "legal_entity", "direction", "engagement_score"]
]

engagement_df.head()

,year,legal_entity,direction,engagement_score
0,2023,CACEIS,BUT - Business Units & Tech,2.50
1,2023,CACEIS,BUT - Inf System Sec & Resil,5.00
2,2023,CACEIS,BUT - Market Solutions,5.00
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,5.00
4,2023,CACEIS,COV - Client & Bus Dev Support,1.25


In [128]:
re_df = abs_df.merge(
    engagement_df,
    on=["year", "legal_entity", "direction"],
    how="left"
)

re_df["re_score"] = re_df["engagement_score"] / re_df["absenteeism_rate"]

re_df = re_df.replace([np.inf, -np.inf], np.nan)

re_df = re_df[
    ["year", "legal_entity", "direction", "re_score"]
]

re_df.head()

,year,legal_entity,direction,re_score
0,2025,CACEIS,BUT - Business Units & Tech,NaN
1,2025,CACEIS,BUT - Gen Secretary & Controls,NaN
2,2025,CACEIS,BUT - Inf System Sec & Resil,30.555556
3,2025,CACEIS,BUT - Information Technology,42.307692
4,2025,CACEIS,BUT - Market Solutions,33.333333


In [129]:
re_df["re_score"].describe()

count    42.000000
mean     12.945255
std       9.062326
min       1.954997
25%       7.523151
50%      10.090509
75%      12.633202
max      42.307692
Name: re_score, dtype: float64

In [130]:
re_df = abs_df.merge(
    engagement_df,
    on=["year", "legal_entity", "direction"],
    how="left"
)

# éviter division par 0
re_df["absenteeism_rate"] = re_df["absenteeism_rate"].replace(0, np.nan)

re_df["re_score"] = (
    re_df["engagement_score"] / re_df["absenteeism_rate"]
)

# fallback si engagement manquant
re_df["re_score"] = re_df["re_score"].fillna(re_df["engagement_score"])

re_df = re_df.replace([np.inf, -np.inf], np.nan)

In [131]:
re_df["re_score"] = re_df["re_score"] / 10

In [132]:
re_df["re_score"].describe()

count    42.000000
mean      1.294526
std       0.906233
min       0.195500
25%       0.752315
50%       1.009051
75%       1.263320
max       4.230769
Name: re_score, dtype: float64

In [134]:
spe_base = finance_tidy.copy()

spe_base["metric"] = spe_base["metric"].str.lower()

In [135]:
strategic_keywords = [
    "training",
    "formation",
    "it",
    "technology",
    "data",
    "esg"
]

spe_base["is_strategic"] = spe_base["metric"].apply(
    lambda x: any(k in x for k in strategic_keywords)
)

In [136]:
spe_df = spe_base.groupby(
    ["year", "legal_entity"]
).agg(
    strategic_costs=("value", lambda x: x[spe_base.loc[x.index, "is_strategic"]].sum()),
    total_costs=("value", "sum")
).reset_index()

spe_df["spe"] = spe_df["strategic_costs"] / spe_df["total_costs"]

spe_df["direction"] = "All Directions"

spe_df = spe_df[
    ["year", "legal_entity", "direction", "spe"]
]

spe_df.head()

,year,legal_entity,direction,spe
0,2023,O/W Europe,All Directions,-0.006842
1,2023,Toutes Filiales Conso,All Directions,-0.007140
2,2024,O/W Europe,All Directions,-0.005290
3,2024,Toutes Filiales Conso,All Directions,-0.005670
4,2025,O/W Europe,All Directions,-0.004198


In [138]:
spe_base = finance_tidy.copy()

# on garde uniquement les coûts
cost_keywords = [
    "cost",
    "personnel",
    "charges",
    "training",
    "formation"
]

spe_base = spe_base[
    spe_base["metric"].str.lower().str.contains("|".join(cost_keywords), na=False)
].copy()

# remettre en positif
spe_base["value"] = spe_base["value"].abs()

In [139]:
strategic_keywords = [
    "training",
    "formation",
    "it",
    "technology",
    "data",
    "esg"
]

spe_base["is_strategic"] = spe_base["metric"].str.lower().apply(
    lambda x: any(k in x for k in strategic_keywords)
)

In [140]:
spe_df = spe_base.groupby(
    ["year", "legal_entity"]
).agg(
    strategic_costs=("value", lambda x: x[spe_base.loc[x.index, "is_strategic"]].sum()),
    total_costs=("value", "sum")
).reset_index()

spe_df["spe"] = spe_df["strategic_costs"] / spe_df["total_costs"]

spe_df["direction"] = "All Directions"

spe_df = spe_df[
    ["year", "legal_entity", "direction", "spe"]
]

spe_df

,year,legal_entity,direction,spe
0,2023,O/W Europe,All Directions,0.002070
1,2023,Toutes Filiales Conso,All Directions,0.002119
2,2024,O/W Europe,All Directions,0.001530
3,2024,Toutes Filiales Conso,All Directions,0.001592
4,2025,O/W Europe,All Directions,0.001408
5,2025,Toutes Filiales Conso,All Directions,0.001395


In [141]:
spe_base = finance_tidy.copy()

spe_base["metric"] = spe_base["metric"].str.lower()

# garder uniquement coûts RH
spe_base = spe_base[
    spe_base["metric"].str.contains("personnel|rémunération|charges", na=False)
].copy()

# remettre positif
spe_base["value"] = spe_base["value"].abs()

In [142]:
strategic_keywords = [
    "it",
    "technology",
    "data",
    "esg",
    "training",
    "formation"
]

spe_base["is_strategic"] = spe_base["metric"].apply(
    lambda x: any(k in x for k in strategic_keywords)
)

In [143]:
spe_df = spe_base.groupby(
    ["year", "legal_entity"]
).agg(
    strategic_payroll=("value", lambda x: x[spe_base.loc[x.index, "is_strategic"]].sum()),
    total_payroll=("value", "sum")
).reset_index()

spe_df["spe"] = spe_df["strategic_payroll"] / spe_df["total_payroll"]

spe_df["direction"] = "All Directions"

spe_df = spe_df[
    ["year", "legal_entity", "direction", "spe"]
]

spe_df

,year,legal_entity,direction,spe
0,2023,O/W Europe,All Directions,0.0
1,2023,Toutes Filiales Conso,All Directions,0.0
2,2024,O/W Europe,All Directions,0.0
3,2024,Toutes Filiales Conso,All Directions,0.0
4,2025,O/W Europe,All Directions,0.0
5,2025,Toutes Filiales Conso,All Directions,0.0


In [145]:
# proxy : investissement stratégique = training / total personnel

training_costs = finance_tidy[
    finance_tidy["metric"].str.contains("training|formation", case=False, na=False)
].copy()

training_costs["value"] = training_costs["value"].abs()

total_personnel = finance_tidy[
    finance_tidy["metric"].str.contains("personnel|rémunération", case=False, na=False)
].copy()

total_personnel["value"] = total_personnel["value"].abs()

In [146]:
training_agg = training_costs.groupby(
    ["year", "legal_entity"]
)["value"].sum().reset_index(name="training_cost")

personnel_agg = total_personnel.groupby(
    ["year", "legal_entity"]
)["value"].sum().reset_index(name="total_personnel")

spe_df = training_agg.merge(
    personnel_agg,
    on=["year", "legal_entity"],
    how="left"
)

spe_df["spe"] = spe_df["training_cost"] / spe_df["total_personnel"]

spe_df["direction"] = "All Directions"

spe_df = spe_df[
    ["year", "legal_entity", "direction", "spe"]
]

spe_df

,year,legal_entity,direction,spe
0,2023,O/W Europe,All Directions,0.005238
1,2023,Toutes Filiales Conso,All Directions,0.005260
2,2024,O/W Europe,All Directions,0.003889
3,2024,Toutes Filiales Conso,All Directions,0.003896
4,2025,O/W Europe,All Directions,0.003420
5,2025,Toutes Filiales Conso,All Directions,0.003255


In [147]:
spe_df["spe"].describe()

count    6.000000
mean     0.004160
std      0.000881
min      0.003255
25%      0.003537
50%      0.003893
75%      0.004903
max      0.005260
Name: spe, dtype: float64

In [149]:
final_df = final_df.merge(
    spe_df,
    on=["year", "legal_entity", "direction"],
    how="left"
)

In [150]:
final_df[["year", "legal_entity", "direction", "spe"]].head()

,year,legal_entity,direction,spe
0,2023,O/W Europe,All Directions,0.005238
1,2023,Toutes Filiales Conso,All Directions,0.005260
2,2024,O/W Europe,All Directions,0.003889
3,2024,Toutes Filiales Conso,All Directions,0.003896
4,2025,O/W Europe,All Directions,0.003420


In [151]:
final_df = final_df.drop(columns=["spe"], errors="ignore")

final_df = final_df.merge(
    spe_df[["year", "legal_entity", "spe"]],
    on=["year", "legal_entity"],
    how="left"
)

In [153]:
final_df["spe_score"] = np.clip(final_df["spe"] / 0.05, 0, 1)

In [155]:
# Vérifier les colonnes disponibles
print(final_df.columns.tolist())

['year', 'legal_entity', 'direction', 'hcva', 'kti', 'skill_decay', 'absent_days', 'employees_absent', 'absenteeism_rate', 'engagement_score', 're_score', 'spe', 'spe_score']


In [156]:
final_df["hcva_score"] = np.clip(final_df["hcva"] / 150, 0, 1)
final_df["kti_score"] = np.clip(final_df["kti"] / 0.8, 0, 1)
final_df["skill_score"] = np.clip(1 - final_df["skill_decay"] / 0.1, 0, 1)
final_df["re_score_norm"] = np.clip(final_df["re_score"] / 4.2, 0, 1)

# tu as déjà spe_score, mais on le sécurise
final_df["spe_score"] = np.clip(final_df["spe"] / 0.05, 0, 1)

In [157]:
final_df["chhi"] = 100 * (
    0.30 * final_df["hcva_score"] +
    0.20 * final_df["kti_score"] +
    0.20 * final_df["skill_score"] +
    0.15 * final_df["re_score_norm"] +
    0.15 * final_df["spe_score"]
)

final_df[[
    "year", "legal_entity", "direction",
    "hcva", "kti", "skill_decay", "re_score", "spe", "chhi"
]].head()

,year,legal_entity,direction,hcva,kti,skill_decay,re_score,spe,chhi
0,2023,O/W Europe,All Directions,186.477702,NaN,NaN,NaN,0.005238,NaN
1,2023,Toutes Filiales Conso,All Directions,158.368498,NaN,NaN,NaN,0.005260,NaN
2,2024,O/W Europe,All Directions,225.456813,NaN,NaN,NaN,0.003889,NaN
3,2024,Toutes Filiales Conso,All Directions,190.082728,NaN,NaN,NaN,0.003896,NaN
4,2025,O/W Europe,All Directions,240.553225,NaN,NaN,NaN,0.003420,NaN


In [158]:
# Base directionnelle principale
final_df = kti_df[["year", "legal_entity", "direction", "kti"]].copy()

final_df = final_df.merge(
    skill_decay_df,
    on=["year", "legal_entity", "direction"],
    how="left"
)

final_df = final_df.merge(
    re_df[["year", "legal_entity", "direction", "re_score"]],
    on=["year", "legal_entity", "direction"],
    how="left"
)

# HCVA au niveau entité
hcva_entity = hcva_df[["year", "legal_entity", "hcva"]].drop_duplicates()

final_df = final_df.merge(
    hcva_entity,
    on=["year", "legal_entity"],
    how="left"
)

# SPE au niveau entité
spe_entity = spe_df[["year", "legal_entity", "spe"]].drop_duplicates()

final_df = final_df.merge(
    spe_entity,
    on=["year", "legal_entity"],
    how="left"
)

final_df.head()

,year,legal_entity,direction,kti,skill_decay,re_score,hcva,spe
0,2023,CACEIS,BUT - Business Units & Tech,0.50,0.0,NaN,NaN,NaN
1,2023,CACEIS,BUT - Inf System Sec & Resil,1.00,0.0,NaN,NaN,NaN
2,2023,CACEIS,BUT - Market Solutions,1.00,0.0,NaN,NaN,NaN
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,1.00,0.0,NaN,NaN,NaN
4,2023,CACEIS,COV - Client & Bus Dev Support,0.25,0.0,NaN,NaN,NaN


In [159]:
final_df["hcva_score"] = np.clip(final_df["hcva"] / 150, 0, 1)
final_df["kti_score"] = np.clip(final_df["kti"] / 0.8, 0, 1)
final_df["skill_score"] = np.clip(1 - final_df["skill_decay"] / 0.1, 0, 1)
final_df["re_score_norm"] = np.clip(final_df["re_score"] / 4.2, 0, 1)
final_df["spe_score"] = np.clip(final_df["spe"] / 0.05, 0, 1)

score_cols = ["hcva_score", "kti_score", "skill_score", "re_score_norm", "spe_score"]
final_df[score_cols] = final_df[score_cols].fillna(0)

In [160]:
final_df["chhi"] = 100 * (
    0.30 * final_df["hcva_score"] +
    0.20 * final_df["kti_score"] +
    0.20 * final_df["skill_score"] +
    0.15 * final_df["re_score_norm"] +
    0.15 * final_df["spe_score"]
)

final_df[[
    "year", "legal_entity", "direction",
    "hcva", "kti", "skill_decay", "re_score", "spe", "chhi"
]].head()

,year,legal_entity,direction,hcva,kti,skill_decay,re_score,spe,chhi
0,2023,CACEIS,BUT - Business Units & Tech,NaN,0.50,0.0,NaN,NaN,32.50
1,2023,CACEIS,BUT - Inf System Sec & Resil,NaN,1.00,0.0,NaN,NaN,40.00
2,2023,CACEIS,BUT - Market Solutions,NaN,1.00,0.0,NaN,NaN,40.00
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,NaN,1.00,0.0,NaN,NaN,40.00
4,2023,CACEIS,COV - Client & Bus Dev Support,NaN,0.25,0.0,NaN,NaN,26.25


In [163]:
# Créer des moyennes globales par année pour HCVA et SPE
hcva_year = hcva_df.groupby("year")["hcva"].mean().reset_index(name="hcva_global")
spe_year = spe_df.groupby("year")["spe"].mean().reset_index(name="spe_global")

# Ajouter les valeurs globales au final_df
final_df = final_df.merge(hcva_year, on="year", how="left")
final_df = final_df.merge(spe_year, on="year", how="left")

# Remplir HCVA/SPE manquants avec fallback annuel
final_df["hcva"] = final_df["hcva"].fillna(final_df["hcva_global"])
final_df["spe"] = final_df["spe"].fillna(final_df["spe_global"])

# Nettoyer colonnes temporaires
final_df = final_df.drop(columns=["hcva_global", "spe_global"])

In [286]:
final_df["hcva_score"] = np.clip(final_df["hcva"] / 150, 0, 1)
final_df["kti_score"] = np.clip(final_df["kti"] / 0.8, 0, 1)
final_df["skill_score"] = np.clip(1 - final_df["skill_decay"] / 0.1, 0, 1)
final_df["re_score_norm"] = np.clip(final_df["re_score"] / 4.2, 0, 1)
final_df["spe_score"] = np.clip(final_df["spe"] / 0.05, 0, 1)

score_cols = ["hcva_score", "kti_score", "skill_score", "re_score_norm", "spe_score"]
final_df[score_cols] = final_df[score_cols].fillna(0)

final_df["chhi"] = 100 * (
    0.30 * final_df["hcva_score"] +
    0.20 * final_df["kti_score"] +
    0.20 * final_df["skill_score"] +
    0.15 * final_df["re_score_norm"] +
    0.15 * final_df["spe_score"]
)

final_df[[
    "year", "legal_entity", "direction",
    "hcva", "kti", "skill_decay", "re_score", "spe", "chhi"
]].head()

,year,legal_entity,direction,hcva,kti,skill_decay,re_score,spe,chhi
0,2023,CACEIS,BUT - Business Units & Tech,172.4231,0.50,0.0,NaN,0.005249,64.074735
1,2023,CACEIS,BUT - Inf System Sec & Resil,172.4231,1.00,0.0,NaN,0.005249,71.574735
2,2023,CACEIS,BUT - Market Solutions,172.4231,1.00,0.0,NaN,0.005249,71.574735
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,172.4231,1.00,0.0,NaN,0.005249,71.574735
4,2023,CACEIS,COV - Client & Bus Dev Support,172.4231,0.25,0.0,NaN,0.005249,57.824735


In [165]:
# Remplacer RE-Score manquant par la moyenne annuelle
re_year = final_df.groupby("year")["re_score"].mean().reset_index(name="re_score_global")

final_df = final_df.merge(re_year, on="year", how="left")

final_df["re_score"] = final_df["re_score"].fillna(final_df["re_score_global"])

final_df = final_df.drop(columns=["re_score_global"])

In [290]:
final_df["hcva_norm"] = np.clip((final_df["hcva"] / 150) * 100, 0, 100)
final_df["kti_norm"] = np.clip(final_df["kti"] * 100, 0, 100)
final_df["skill_norm"] = np.clip(100 - (final_df["skill_decay"] * 100), 0, 100)
final_df["re_norm"] = np.clip((final_df["re_score"] / 4.2) * 100, 0, 100)
final_df["spe_norm"] = np.clip((final_df["spe"] / 0.30) * 100, 0, 100)

final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
).round(2)

In [291]:

final_df[[
    "year", "legal_entity", "direction",
    "hcva", "kti", "skill_decay", "re_score", "spe", "chhi"
]].head()

,year,legal_entity,direction,hcva,kti,skill_decay,re_score,spe,chhi
0,2023,CACEIS,BUT - Business Units & Tech,172.4231,0.50,0.0,0.0,0.00526,60.26
1,2023,CACEIS,BUT - Inf System Sec & Resil,172.4231,1.00,0.0,0.0,0.00526,70.26
2,2023,CACEIS,BUT - Market Solutions,172.4231,1.00,0.0,0.0,0.00526,70.26
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,172.4231,1.00,0.0,0.0,0.00526,70.26
4,2023,CACEIS,COV - Client & Bus Dev Support,172.4231,0.25,0.0,0.0,0.00526,55.26


In [179]:
final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
)

final_df["chhi"] = final_df["chhi"].round(2)

In [180]:
# ============================================================
# FIX RE-SCORE MERGE + RECALCULATE CHHI
# ============================================================

# 1. Remove old broken RE-score from final_df
final_df = final_df.drop(columns=["re_score"], errors="ignore")

# 2. Make sure re_df has only required columns
re_clean = re_df[["year", "legal_entity", "direction", "re_score"]].copy()

# 3. Merge correct RE-score into final_df
final_df = final_df.merge(
    re_clean,
    on=["year", "legal_entity", "direction"],
    how="left"
)

# 4. Fill missing RE-score with yearly mean
final_df["re_score"] = final_df["re_score"].fillna(
    final_df.groupby("year")["re_score"].transform("mean")
)

# 5. Final fallback
final_df["re_score"] = final_df["re_score"].fillna(0)

# ============================================================
# NORMALIZATION TO 0–100
# ============================================================

final_df["hcva_norm"] = np.clip((final_df["hcva"] / 150) * 100, 0, 100)

final_df["kti_norm"] = np.clip(final_df["kti"] * 100, 0, 100)

final_df["skill_norm"] = np.clip(
    100 - (final_df["skill_decay"] * 100),
    0,
    100
)

final_df["re_norm"] = np.clip(
    (final_df["re_score"] / 4.2) * 100,
    0,
    100
)

final_df["spe_norm"] = np.clip(
    (final_df["spe"] / 0.30) * 100,
    0,
    100
)

# ============================================================
# CHHI INDEX
# ============================================================

final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
).round(2)

# ============================================================
# FINAL DASHBOARD DATASET
# ============================================================

dashboard_df = final_df.rename(columns={
    "year": "Year",
    "legal_entity": "Legal Entity",
    "direction": "Direction",
    "hcva": "HCVA",
    "kti": "KTI",
    "skill_decay": "Skill Decay",
    "re_score": "RE-Score",
    "spe": "SPE",
    "chhi": "CHHI Index"
})

dashboard_df = dashboard_df[
    [
        "Year",
        "Legal Entity",
        "Direction",
        "HCVA",
        "KTI",
        "Skill Decay",
        "RE-Score",
        "SPE",
        "CHHI Index"
    ]
]

dashboard_df.head()

,Year,Legal Entity,Direction,HCVA,KTI,Skill Decay,RE-Score,SPE,CHHI Index
0,2023,CACEIS,BUT - Business Units & Tech,172.4231,0.50,0.0,0.0,0.005249,60.26
1,2023,CACEIS,BUT - Inf System Sec & Resil,172.4231,1.00,0.0,0.0,0.005249,70.26
2,2023,CACEIS,BUT - Market Solutions,172.4231,1.00,0.0,0.0,0.005249,70.26
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,172.4231,1.00,0.0,0.0,0.005249,70.26
4,2023,CACEIS,COV - Client & Bus Dev Support,172.4231,0.25,0.0,0.0,0.005249,55.26


In [308]:
# ============================================================
# EXPORT FINAL CSV POUR STREAMLIT
# ============================================================

dashboard_df = final_df.rename(columns={
    "year": "Year",
    "legal_entity": "Legal Entity",
    "direction": "Direction",
    "hcva": "HCVA",
    "kti": "KTI",
    "skill_decay": "Skill Decay",
    "re_score": "RE-Score",
    "spe": "SPE",
    "chhi": "CHHI Index"
})

dashboard_df = dashboard_df[
    [
        "Year",
        "Legal Entity",
        "Direction",
        "HCVA",
        "KTI",
        "Skill Decay",
        "RE-Score",
        "SPE",
        "CHHI Index"
    ]
]

# Exclure 2022 par sécurité
dashboard_df = dashboard_df[dashboard_df["Year"].isin([2023, 2024, 2025])]

# Arrondir les valeurs
dashboard_df["HCVA"] = dashboard_df["HCVA"].round(2)
dashboard_df["KTI"] = dashboard_df["KTI"].round(3)
dashboard_df["Skill Decay"] = dashboard_df["Skill Decay"].round(3)
dashboard_df["RE-Score"] = dashboard_df["RE-Score"].round(2)
dashboard_df["SPE"] = dashboard_df["SPE"].round(4)
dashboard_df["CHHI Index"] = dashboard_df["CHHI Index"].round(2)

# Export CSV
dashboard_df.to_csv(
    "chhi_dashboard_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

dashboard_df.head()

,Year,Legal Entity,Direction,HCVA,KTI,Skill Decay,RE-Score,SPE,CHHI Index
0,2023,CACEIS,BUT - Business Units & Tech,172.42,0.50,0.0,0.0,0.0053,60.26
1,2023,CACEIS,BUT - Inf System Sec & Resil,172.42,1.00,0.0,0.0,0.0053,70.26
2,2023,CACEIS,BUT - Market Solutions,172.42,1.00,0.0,0.0,0.0053,70.26
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,172.42,1.00,0.0,0.0,0.0053,70.26
4,2023,CACEIS,COV - Client & Bus Dev Support,172.42,0.25,0.0,0.0,0.0053,55.26


In [309]:
# FIX SKILL DECAY
skill_clean = skill_decay_df[
    ["year", "legal_entity", "direction", "skill_decay"]
].drop_duplicates()

final_df = final_df.drop(
    columns=["skill_decay", "skill_decay_x", "skill_decay_y"],
    errors="ignore"
)

final_df = final_df.merge(
    skill_clean,
    on=["year", "legal_entity", "direction"],
    how="left"
)

# fallback intelligent
final_df["skill_decay"] = final_df["skill_decay"].fillna(
    final_df.groupby(["year", "legal_entity"])["skill_decay"].transform("mean")
)

final_df["skill_decay"] = final_df["skill_decay"].fillna(
    final_df.groupby("year")["skill_decay"].transform("mean")
)

final_df["skill_decay"] = final_df["skill_decay"].fillna(0)

In [310]:
# FIX RE-SCORE
re_clean = re_df[
    ["year", "legal_entity", "direction", "re_score"]
].drop_duplicates()

final_df = final_df.drop(
    columns=["re_score", "re_score_x", "re_score_y"],
    errors="ignore"
)

final_df = final_df.merge(
    re_clean,
    on=["year", "legal_entity", "direction"],
    how="left"
)

final_df["re_score"] = final_df["re_score"].fillna(
    final_df.groupby(["year", "legal_entity"])["re_score"].transform("mean")
)

final_df["re_score"] = final_df["re_score"].fillna(
    final_df.groupby("year")["re_score"].transform("mean")
)

final_df["re_score"] = final_df["re_score"].fillna(0)

In [311]:
# NORMALISATION
final_df["hcva_norm"] = np.clip((final_df["hcva"] / 150) * 100, 0, 100)
final_df["kti_norm"] = np.clip(final_df["kti"] * 100, 0, 100)
final_df["skill_norm"] = np.clip(100 - (final_df["skill_decay"] * 100), 0, 100)
final_df["re_norm"] = np.clip((final_df["re_score"] / 4.2) * 100, 0, 100)
final_df["spe_norm"] = np.clip((final_df["spe"] / 0.30) * 100, 0, 100)

# CHHI
final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
).round(2)

In [312]:
final_df[[
    "year",
    "legal_entity",
    "direction",
    "hcva",
    "kti",
    "skill_decay",
    "re_score",
    "spe",
    "chhi"
]].head()

,year,legal_entity,direction,hcva,kti,skill_decay,re_score,spe,chhi
0,2023,CACEIS,BUT - Business Units & Tech,172.4231,0.50,0.0,0.0,0.00526,60.26
1,2023,CACEIS,BUT - Inf System Sec & Resil,172.4231,1.00,0.0,0.0,0.00526,70.26
2,2023,CACEIS,BUT - Market Solutions,172.4231,1.00,0.0,0.0,0.00526,70.26
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,172.4231,1.00,0.0,0.0,0.00526,70.26
4,2023,CACEIS,COV - Client & Bus Dev Support,172.4231,0.25,0.0,0.0,0.00526,55.26


In [313]:
# Clean SPE before merge
spe_clean = spe_df[["year", "legal_entity", "spe"]].drop_duplicates()

# Remove any existing SPE columns
final_df = final_df.drop(columns=["spe", "spe_x", "spe_y"], errors="ignore")

# Merge clean SPE
final_df = final_df.merge(
    spe_clean,
    on=["year", "legal_entity"],
    how="left"
)

In [314]:
final_df[["year", "legal_entity", "direction", "spe"]].head()

,year,legal_entity,direction,spe
0,2023,CACEIS,BUT - Business Units & Tech,NaN
1,2023,CACEIS,BUT - Inf System Sec & Resil,NaN
2,2023,CACEIS,BUT - Market Solutions,NaN
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,NaN
4,2023,CACEIS,COV - Client & Bus Dev Support,NaN


In [315]:
spe_df[["year", "legal_entity", "spe"]].head()

,year,legal_entity,spe
0,2023,O/W Europe,0.005238
1,2023,Toutes Filiales Conso,0.005260
2,2024,O/W Europe,0.003889
3,2024,Toutes Filiales Conso,0.003896
4,2025,O/W Europe,0.003420


In [316]:
def norm_txt(s):
    return (
        s.astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

spe_clean = spe_df[["year", "legal_entity", "spe"]].drop_duplicates().copy()

final_df["_le_key"] = norm_txt(final_df["legal_entity"])
spe_clean["_le_key"] = norm_txt(spe_clean["legal_entity"])

final_df = final_df.drop(columns=["spe", "spe_x", "spe_y"], errors="ignore")

final_df = final_df.merge(
    spe_clean[["year", "_le_key", "spe"]],
    on=["year", "_le_key"],
    how="left"
)

final_df = final_df.drop(columns=["_le_key"], errors="ignore")

final_df[["year", "legal_entity", "direction", "spe"]].head()

,year,legal_entity,direction,spe
0,2023,CACEIS,BUT - Business Units & Tech,NaN
1,2023,CACEIS,BUT - Inf System Sec & Resil,NaN
2,2023,CACEIS,BUT - Market Solutions,NaN
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,NaN
4,2023,CACEIS,COV - Client & Bus Dev Support,NaN


In [317]:
spe_df.groupby(["year", "legal_entity"])["spe"].mean()

year  legal_entity         
2023  O/W Europe               0.005238
      Toutes Filiales Conso    0.005260
2024  O/W Europe               0.003889
      Toutes Filiales Conso    0.003896
2025  O/W Europe               0.003420
      Toutes Filiales Conso    0.003255
Name: spe, dtype: float64

In [318]:
# ============================================================
# FIX SPE: utiliser SPE global "Toutes Filiales Conso"
# ============================================================

spe_global = (
    spe_df[spe_df["legal_entity"].eq("Toutes Filiales Conso")]
    [["year", "spe"]]
    .drop_duplicates()
)

final_df = final_df.drop(columns=["spe", "spe_x", "spe_y"], errors="ignore")

final_df = final_df.merge(
    spe_global,
    on="year",
    how="left"
)

final_df["spe"] = final_df["spe"].fillna(
    final_df.groupby("year")["spe"].transform("mean")
).fillna(0)

final_df[["year", "legal_entity", "direction", "spe"]].head()

,year,legal_entity,direction,spe
0,2023,CACEIS,BUT - Business Units & Tech,0.00526
1,2023,CACEIS,BUT - Inf System Sec & Resil,0.00526
2,2023,CACEIS,BUT - Market Solutions,0.00526
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,0.00526
4,2023,CACEIS,COV - Client & Bus Dev Support,0.00526


In [319]:
final_df["spe_norm"] = np.clip((final_df["spe"] / 0.30) * 100, 0, 100)

In [320]:
# ============================================================
# RECALCULER TOUTES LES COLONNES NORMALISÉES
# ============================================================

final_df["hcva"] = final_df["hcva"].fillna(0)
final_df["kti"] = final_df["kti"].fillna(0)
final_df["skill_decay"] = final_df["skill_decay"].fillna(0)
final_df["re_score"] = final_df["re_score"].fillna(0)
final_df["spe"] = final_df["spe"].fillna(0)

# HCVA target = 150 k€
final_df["hcva_norm"] = np.clip(
    (final_df["hcva"] / 150) * 100,
    0,
    100
)

# KTI déjà en ratio 0–1
final_df["kti_norm"] = np.clip(
    final_df["kti"] * 100,
    0,
    100
)

# Skill Decay inverse
final_df["skill_norm"] = np.clip(
    100 - (final_df["skill_decay"] * 100),
    0,
    100
)

# RE target = 4.2
final_df["re_norm"] = np.clip(
    (final_df["re_score"] / 4.2) * 100,
    0,
    100
)

# SPE target = 30%
final_df["spe_norm"] = np.clip(
    (final_df["spe"] / 0.30) * 100,
    0,
    100
)

# CHHI
final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
).round(2)

dashboard_df = final_df.rename(columns={
    "year": "Year",
    "legal_entity": "Legal Entity",
    "direction": "Direction",
    "hcva": "HCVA",
    "kti": "KTI",
    "skill_decay": "Skill Decay",
    "re_score": "RE-Score",
    "spe": "SPE",
    "chhi": "CHHI Index"
})

dashboard_df = dashboard_df[
    [
        "Year", "Legal Entity", "Direction",
        "HCVA", "KTI", "Skill Decay",
        "RE-Score", "SPE", "CHHI Index"
    ]
]

dashboard_df.head()

,Year,Legal Entity,Direction,HCVA,KTI,Skill Decay,RE-Score,SPE,CHHI Index
0,2023,CACEIS,BUT - Business Units & Tech,172.4231,0.50,0.0,0.0,0.00526,60.26
1,2023,CACEIS,BUT - Inf System Sec & Resil,172.4231,1.00,0.0,0.0,0.00526,70.26
2,2023,CACEIS,BUT - Market Solutions,172.4231,1.00,0.0,0.0,0.00526,70.26
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,172.4231,1.00,0.0,0.0,0.00526,70.26
4,2023,CACEIS,COV - Client & Bus Dev Support,172.4231,0.25,0.0,0.0,0.00526,55.26


In [321]:
final_df[["hcva_norm", "kti_norm", "skill_norm", "re_norm", "spe_norm", "chhi"]].head()

,hcva_norm,kti_norm,skill_norm,re_norm,spe_norm,chhi
0,100.0,50.0,100.0,0.0,1.753247,60.26
1,100.0,100.0,100.0,0.0,1.753247,70.26
2,100.0,100.0,100.0,0.0,1.753247,70.26
3,100.0,100.0,100.0,0.0,1.753247,70.26
4,100.0,25.0,100.0,0.0,1.753247,55.26


In [322]:
# ============================================================
# FIX HCVA + RE-SCORE AVANT CHHI
# ============================================================

# ---------- HCVA : fallback par année ----------
hcva_clean = (
    hcva_df[["year", "legal_entity", "hcva"]]
    .drop_duplicates()
    .copy()
)

# Si hcva_df ne matche pas CACEIS, on prend le HCVA moyen annuel
hcva_year = (
    hcva_clean
    .groupby("year", as_index=False)["hcva"]
    .mean()
)

final_df = final_df.drop(columns=["hcva", "hcva_x", "hcva_y"], errors="ignore")

final_df = final_df.merge(
    hcva_year,
    on="year",
    how="left"
)

final_df["hcva"] = final_df["hcva"].fillna(0)


# ---------- RE-SCORE : fallback par année ----------
re_clean = (
    re_df[["year", "legal_entity", "direction", "re_score"]]
    .drop_duplicates()
    .copy()
)

re_year = (
    re_clean
    .groupby("year", as_index=False)["re_score"]
    .mean()
)

final_df = final_df.drop(columns=["re_score", "re_score_x", "re_score_y"], errors="ignore")

final_df = final_df.merge(
    re_year,
    on="year",
    how="left"
)

final_df["re_score"] = final_df["re_score"].fillna(0)


# ---------- Vérification ----------
final_df[["year", "legal_entity", "direction", "hcva", "re_score", "spe"]].head()

,year,legal_entity,direction,hcva,re_score,spe
0,2023,CACEIS,BUT - Business Units & Tech,172.4231,0.0,0.00526
1,2023,CACEIS,BUT - Inf System Sec & Resil,172.4231,0.0,0.00526
2,2023,CACEIS,BUT - Market Solutions,172.4231,0.0,0.00526
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,172.4231,0.0,0.00526
4,2023,CACEIS,COV - Client & Bus Dev Support,172.4231,0.0,0.00526


In [323]:
final_df["hcva_norm"] = np.clip((final_df["hcva"] / 150) * 100, 0, 100)
final_df["kti_norm"] = np.clip(final_df["kti"] * 100, 0, 100)
final_df["skill_norm"] = np.clip(100 - (final_df["skill_decay"] * 100), 0, 100)
final_df["re_norm"] = np.clip((final_df["re_score"] / 4.2) * 100, 0, 100)
final_df["spe_norm"] = np.clip((final_df["spe"] / 0.30) * 100, 0, 100)

final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
).round(2)

In [324]:
# Vérifier les valeurs utilisées
final_df["hcva_norm"] = np.clip((final_df["hcva"] / 150) * 100, 0, 100)
final_df["kti_norm"] = np.clip(final_df["kti"] * 100, 0, 100)
final_df["skill_norm"] = np.clip(100 - (final_df["skill_decay"] * 100), 0, 100)
final_df["re_norm"] = np.clip((final_df["re_score"] / 4.2) * 100, 0, 100)
final_df["spe_norm"] = np.clip((final_df["spe"] / 0.30) * 100, 0, 100)

final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
).round(2)

dashboard_df = final_df.rename(columns={
    "year": "Year",
    "legal_entity": "Legal Entity",
    "direction": "Direction",
    "hcva": "HCVA",
    "kti": "KTI",
    "skill_decay": "Skill Decay",
    "re_score": "RE-Score",
    "spe": "SPE",
    "chhi": "CHHI Index"
})

dashboard_df = dashboard_df[
    ["Year", "Legal Entity", "Direction", "HCVA", "KTI",
     "Skill Decay", "RE-Score", "SPE", "CHHI Index"]
]

dashboard_df.head()

,Year,Legal Entity,Direction,HCVA,KTI,Skill Decay,RE-Score,SPE,CHHI Index
0,2023,CACEIS,BUT - Business Units & Tech,172.4231,0.50,0.0,0.0,0.00526,60.26
1,2023,CACEIS,BUT - Inf System Sec & Resil,172.4231,1.00,0.0,0.0,0.00526,70.26
2,2023,CACEIS,BUT - Market Solutions,172.4231,1.00,0.0,0.0,0.00526,70.26
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,172.4231,1.00,0.0,0.0,0.00526,70.26
4,2023,CACEIS,COV - Client & Bus Dev Support,172.4231,0.25,0.0,0.0,0.00526,55.26


In [325]:
# ============================================================
# RE-SCORE CALCULATION
# ============================================================

# Engagement depuis KTI (proxy)
engagement_df = kti_df.copy()
engagement_df["engagement_score"] = engagement_df["kti"] * 5

# Merge avec absentéisme
re_df = abs_df.merge(
    engagement_df,
    on=["year", "legal_entity", "direction"],
    how="left"
)

# Nettoyage
re_df["absenteeism_rate"] = re_df["absenteeism_rate"].replace(0, np.nan)

# Calcul
re_df["re_score"] = (
    re_df["engagement_score"] / re_df["absenteeism_rate"]
)

# Nettoyage valeurs extrêmes
re_df["re_score"] = re_df["re_score"].replace([np.inf, -np.inf], np.nan)

# Fallback propre
re_df["re_score"] = re_df["re_score"].fillna(
    re_df.groupby("year")["re_score"].transform("mean")
)

re_df["re_score"] = re_df["re_score"].fillna(0)

re_df.head()

,year,legal_entity,direction,absent_days,employees_absent,absenteeism_rate,kti,utilization_score,response_rate,engagement_score,re_score
0,2025,CACEIS,BUT - Business Units & Tech,36.0,1,0.163636,NaN,NaN,NaN,NaN,12.945255
1,2025,CACEIS,BUT - Gen Secretary & Controls,29.5,1,0.134091,NaN,NaN,NaN,NaN,12.945255
2,2025,CACEIS,BUT - Inf System Sec & Resil,36.0,1,0.163636,1.0,1.0,1.0,5.0,30.555556
3,2025,CACEIS,BUT - Information Technology,26.0,1,0.118182,1.0,1.0,1.0,5.0,42.307692
4,2025,CACEIS,BUT - Market Solutions,33.0,1,0.150000,1.0,1.0,1.0,5.0,33.333333


In [326]:
final_df = final_df.drop(columns=["skill_decay", "re_score"], errors="ignore")

final_df = final_df.merge(skill_decay_df, on=["year","legal_entity","direction"], how="left")
final_df = final_df.merge(re_df[["year","legal_entity","direction","re_score"]],
                          on=["year","legal_entity","direction"], how="left")

In [329]:
# ============================================================
# FIX SKILL DECAY MERGE
# ============================================================

skill_clean = skill_decay_df[
    ["year", "legal_entity", "direction", "skill_decay"]
].drop_duplicates()

final_df = final_df.drop(
    columns=["skill_decay", "skill_decay_x", "skill_decay_y"],
    errors="ignore"
)

final_df = final_df.merge(
    skill_clean,
    on=["year", "legal_entity", "direction"],
    how="left"
)

# fallback si certaines directions ne matchent pas
final_df["skill_decay"] = final_df["skill_decay"].fillna(
    final_df.groupby(["year", "legal_entity"])["skill_decay"].transform("mean")
)

final_df["skill_decay"] = final_df["skill_decay"].fillna(
    final_df.groupby("year")["skill_decay"].transform("mean")
)

final_df["skill_decay"] = final_df["skill_decay"].fillna(0)

final_df[["year", "legal_entity", "direction", "skill_decay"]].sort_values(
    "skill_decay", ascending=False
).head(10)

,year,legal_entity,direction,skill_decay
92,2024,CACEIS Bank,COMPLIANCE,0.750000
103,2024,CACEIS Bank,OPERATIONS,0.565217
115,2024,CACEIS Bank,STI - ESG,0.500000
84,2024,CACEIS Bank,BUT - Business Units & Tech,0.500000
75,2024,CACEIS,LEGAL,0.500000
134,2025,CACEIS,STI - 3D & Products,0.500000
136,2025,CACEIS Bank,BUT - Business Units & Tech,0.500000
121,2024,CACEIS Fund Administration,OPERATIONS,0.450000
129,2025,CACEIS,Human Resources,0.428571
91,2024,CACEIS Bank,CLIENT & BUSINESS DEVELOPMENT SUPPORT,0.400000


In [330]:
# ============================================================
# FIX RE-SCORE MERGE
# ============================================================

re_clean = re_df[
    ["year", "legal_entity", "direction", "re_score"]
].drop_duplicates()

final_df = final_df.drop(
    columns=["re_score", "re_score_x", "re_score_y"],
    errors="ignore"
)

final_df = final_df.merge(
    re_clean,
    on=["year", "legal_entity", "direction"],
    how="left"
)

final_df["re_score"] = final_df["re_score"].fillna(
    final_df.groupby(["year", "legal_entity"])["re_score"].transform("mean")
)

final_df["re_score"] = final_df["re_score"].fillna(
    final_df.groupby("year")["re_score"].transform("mean")
)

final_df["re_score"] = final_df["re_score"].fillna(0)

final_df[["year", "legal_entity", "direction", "re_score"]].sort_values(
    "re_score", ascending=False
).head(10)

,year,legal_entity,direction,re_score
125,2025,CACEIS,BUT - Information Technology,42.307692
126,2025,CACEIS,BUT - Market Solutions,33.333333
134,2025,CACEIS,STI - 3D & Products,32.835821
124,2025,CACEIS,BUT - Inf System Sec & Resil,30.555556
136,2025,CACEIS Bank,BUT - Business Units & Tech,26.347305
128,2025,CACEIS,General Management,23.790214
146,2025,CACEIS Bank,COV - General Secretary,22.701215
145,2025,CACEIS Bank,COV - Coverage France,22.197982
154,2025,CACEIS Bank,SPF - Procurement,18.839761
162,2025,CACEIS Fund Administration,BUT - Information Technology,14.221780


In [331]:
# ============================================================
# FALLBACK ROBUSTE SKILL DECAY + RE-SCORE
# ============================================================

# Skill Decay fallback depuis skill_decay_df
skill_year = (
    skill_decay_df
    .groupby("year", as_index=False)["skill_decay"]
    .mean()
    .rename(columns={"skill_decay": "skill_decay_year"})
)

final_df = final_df.merge(skill_year, on="year", how="left")

final_df["skill_decay"] = np.where(
    final_df["skill_decay"].isna() | (final_df["skill_decay"] == 0),
    final_df["skill_decay_year"],
    final_df["skill_decay"]
)

final_df = final_df.drop(columns=["skill_decay_year"], errors="ignore")


# RE-Score fallback depuis re_df
re_year = (
    re_df
    .groupby("year", as_index=False)["re_score"]
    .mean()
    .rename(columns={"re_score": "re_score_year"})
)

final_df = final_df.merge(re_year, on="year", how="left")

final_df["re_score"] = np.where(
    final_df["re_score"].isna() | (final_df["re_score"] == 0),
    final_df["re_score_year"],
    final_df["re_score"]
)

final_df = final_df.drop(columns=["re_score_year"], errors="ignore")

final_df[[
    "year", "legal_entity", "direction",
    "skill_decay", "re_score"
]].head()

,year,legal_entity,direction,skill_decay,re_score
0,2023,CACEIS,BUT - Business Units & Tech,0.0,NaN
1,2023,CACEIS,BUT - Inf System Sec & Resil,0.0,NaN
2,2023,CACEIS,BUT - Market Solutions,0.0,NaN
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,0.0,NaN
4,2023,CACEIS,COV - Client & Bus Dev Support,0.0,NaN


In [332]:
final_df["hcva_norm"] = np.clip((final_df["hcva"] / 150) * 100, 0, 100)
final_df["kti_norm"] = np.clip(final_df["kti"] * 100, 0, 100)
final_df["skill_norm"] = np.clip(100 - (final_df["skill_decay"] * 100), 0, 100)
final_df["re_norm"] = np.clip((final_df["re_score"] / 4.2) * 100, 0, 100)
final_df["spe_norm"] = np.clip((final_df["spe"] / 0.30) * 100, 0, 100)

final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
).round(2)

In [333]:
# ============================================================
# FIX FINAL RE-SCORE (ROBUSTE)
# ============================================================

# moyenne annuelle
re_year = (
    re_df
    .groupby("year", as_index=False)["re_score"]
    .mean()
    .rename(columns={"re_score": "re_score_year"})
)

# merge
final_df = final_df.merge(re_year, on="year", how="left")

# fallback intelligent
final_df["re_score"] = np.where(
    final_df["re_score"].isna(),
    final_df["re_score_year"],
    final_df["re_score"]
)

# fallback ultime
final_df["re_score"] = final_df["re_score"].fillna(0)

# clean
final_df = final_df.drop(columns=["re_score_year"], errors="ignore")

In [334]:
# ============================================================
# FIX FINAL SKILL DECAY (ROBUSTE)
# ============================================================

skill_year = (
    skill_decay_df
    .groupby("year", as_index=False)["skill_decay"]
    .mean()
    .rename(columns={"skill_decay": "skill_decay_year"})
)

final_df = final_df.merge(skill_year, on="year", how="left")

final_df["skill_decay"] = np.where(
    (final_df["skill_decay"].isna()) | (final_df["skill_decay"] == 0),
    final_df["skill_decay_year"],
    final_df["skill_decay"]
)

final_df["skill_decay"] = final_df["skill_decay"].fillna(0)

final_df = final_df.drop(columns=["skill_decay_year"], errors="ignore")

In [335]:
final_df["hcva_norm"] = np.clip((final_df["hcva"] / 150) * 100, 0, 100)
final_df["kti_norm"] = np.clip(final_df["kti"] * 100, 0, 100)
final_df["skill_norm"] = np.clip(100 - (final_df["skill_decay"] * 100), 0, 100)
final_df["re_norm"] = np.clip((final_df["re_score"] / 4.2) * 100, 0, 100)
final_df["spe_norm"] = np.clip((final_df["spe"] / 0.30) * 100, 0, 100)

final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
).round(2)

In [336]:
re_df.groupby("year")["re_score"].describe()

,count,mean,std,min,25%,50%,75%,max
year,,,,,,,,
2025,54.0,12.945255,7.970649,1.954997,8.306248,11.918493,12.945255,42.307692


In [338]:
# ============================================================
# FIX RE-SCORE : fallback global si année sans données
# ============================================================

re_global_mean = re_df["re_score"].replace(0, np.nan).mean()

final_df["re_score"] = final_df["re_score"].replace(0, np.nan)

final_df["re_score"] = final_df["re_score"].fillna(
    final_df.groupby("year")["re_score"].transform("mean")
)

final_df["re_score"] = final_df["re_score"].fillna(re_global_mean)

final_df["re_score"] = final_df["re_score"].fillna(0)

final_df[["year", "legal_entity", "direction", "skill_decay", "re_score"]].head(20)

,year,legal_entity,direction,skill_decay,re_score
0,2023,CACEIS,BUT - Business Units & Tech,0.0,12.945255
1,2023,CACEIS,BUT - Inf System Sec & Resil,0.0,12.945255
2,2023,CACEIS,BUT - Market Solutions,0.0,12.945255
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,0.0,12.945255
4,2023,CACEIS,COV - Client & Bus Dev Support,0.0,12.945255
5,2023,CACEIS,COV - PERES,0.0,12.945255
6,2023,CACEIS,FINANCE AND ADMINISTRATION,0.0,12.945255
7,2023,CACEIS,GENERAL INSPECTION,0.0,12.945255
8,2023,CACEIS,General Management,0.0,12.945255
9,2023,CACEIS,Human Resources,0.0,12.945255


In [339]:
final_df["hcva_norm"] = np.clip((final_df["hcva"] / 150) * 100, 0, 100)
final_df["kti_norm"] = np.clip(final_df["kti"] * 100, 0, 100)
final_df["skill_norm"] = np.clip(100 - (final_df["skill_decay"] * 100), 0, 100)
final_df["re_norm"] = np.clip((final_df["re_score"] / 4.2) * 100, 0, 100)
final_df["spe_norm"] = np.clip((final_df["spe"] / 0.30) * 100, 0, 100)

final_df["chhi"] = (
    0.30 * final_df["hcva_norm"] +
    0.20 * final_df["kti_norm"] +
    0.20 * final_df["skill_norm"] +
    0.15 * final_df["re_norm"] +
    0.15 * final_df["spe_norm"]
).round(2)

In [340]:
final_df["re_score"] = final_df["re_score"].fillna(0)

In [341]:
# ============================================================
# EXPORT FINAL CSV POUR STREAMLIT
# ============================================================

dashboard_df = final_df.rename(columns={
    "year": "Year",
    "legal_entity": "Legal Entity",
    "direction": "Direction",
    "hcva": "HCVA",
    "kti": "KTI",
    "skill_decay": "Skill Decay",
    "re_score": "RE-Score",
    "spe": "SPE",
    "chhi": "CHHI Index"
})

dashboard_df = dashboard_df[
    [
        "Year",
        "Legal Entity",
        "Direction",
        "HCVA",
        "KTI",
        "Skill Decay",
        "RE-Score",
        "SPE",
        "CHHI Index"
    ]
]

# Exclure 2022 par sécurité
dashboard_df = dashboard_df[dashboard_df["Year"].isin([2023, 2024, 2025])]

# Arrondir les valeurs
dashboard_df["HCVA"] = dashboard_df["HCVA"].round(2)
dashboard_df["KTI"] = dashboard_df["KTI"].round(3)
dashboard_df["Skill Decay"] = dashboard_df["Skill Decay"].round(3)
dashboard_df["RE-Score"] = dashboard_df["RE-Score"].round(2)
dashboard_df["SPE"] = dashboard_df["SPE"].round(4)
dashboard_df["CHHI Index"] = dashboard_df["CHHI Index"].round(2)

# Export CSV
dashboard_df.to_csv(
    "chhi_dashboard_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

dashboard_df.head()

,Year,Legal Entity,Direction,HCVA,KTI,Skill Decay,RE-Score,SPE,CHHI Index
0,2023,CACEIS,BUT - Business Units & Tech,172.42,0.50,0.0,12.95,0.0053,75.26
1,2023,CACEIS,BUT - Inf System Sec & Resil,172.42,1.00,0.0,12.95,0.0053,85.26
2,2023,CACEIS,BUT - Market Solutions,172.42,1.00,0.0,12.95,0.0053,85.26
3,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,172.42,1.00,0.0,12.95,0.0053,85.26
4,2023,CACEIS,COV - Client & Bus Dev Support,172.42,0.25,0.0,12.95,0.0053,70.26
